Note book for an example of a variational auto-encoder

In [1]:
from mach3pythonutils.utils import setup_logging
from mach3pythonutils.ml_algorithms.implementations.torch.torch_algorithm import TorchVAEAlgorithm
from mach3pythonutils.ml_algorithms.implementations.torch.torch_models import TorchVAEModel, TorchPerceptronModel
from mach3pythonutils.file_io.root_dataset import ROOTDataset

import torch

setup_logging(log_level="INFO")

In [2]:
import gdown
from pathlib import Path
# Download file from google drive

file_url="https://drive.google.com/file/d/1iE6xFhn3BH_HnLUfQ7KFGy2wfeH52Rwf/view?usp=sharing"

# download the file
INPUT_FILE = Path("../models/demo_chain.root")

if not INPUT_FILE.exists():
    # download the file
    INPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(file_url, str(INPUT_FILE), quiet=False, fuzzy=True)


In [3]:
# Set up file handler

TREE_NAME = "posteriors"
INPUT_FILE = "../models/demo_chain.root"

DATASET = ROOTDataset(INPUT_FILE, tree_name=TREE_NAME, branches=["xsec_1","xsec_2", "xsec_3"], labels=["xsec_1","xsec_2", "xsec_3"], only_unique_entries=False)

[2025-08-08 11:28:05] INFO     ┌────────────────────────────┐                                          ]8;id=705182;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=67165;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#111\111]8;;\

                      INFO     │ Initializing ROOTDataset │                                            ]8;id=709620;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=154813;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#112\112]8;;\

                      INFO     └────────────────────────────┘                                          ]8;id=283458;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=461041;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#113\113]8;;\

                      INFO     ℹ️ Opening ROOT file: ../models/demo_chain.root                          ]8;id=30305;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=636810;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#169\169]8;;\

                      INFO     ℹ️ Tree 'posteriors' has 99999 entries                                   ]8;id=225360;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=370155;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#169\169]8;;\

                      INFO     ✅ Initialized ROOTDataset with 99999 entries, branches: ['xsec_1',     ]8;id=74761;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=545952;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#125\125]8;;\
                               'xsec_2', 'xsec_3'], labels: ['xsec_1', 'xsec_2', 'xsec_3']                         

In [4]:
#Machine learning time

# Dimensions
INPUT_DIM = DATASET.get_n_features()
OUTPUT_DIM = DATASET.get_n_labels()
LATENT_DIM = 10

In [5]:
# Layers

ENCODE_LAYERS = [
    torch.nn.Linear(INPUT_DIM, 20),
    torch.nn.LeakyReLU(0.2),
    torch.nn.Linear(20, 15),
    torch.nn.LeakyReLU(0.2),
    torch.nn.Linear(15, LATENT_DIM),
]

ENCODER_MODEL = TorchPerceptronModel(INPUT_DIM, LATENT_DIM, ENCODE_LAYERS)

DECODE_LAYERS = [
    torch.nn.Linear(2, LATENT_DIM),
    torch.nn.LeakyReLU(0.2),
    torch.nn.Linear(LATENT_DIM, 15),
    torch.nn.LeakyReLU(0.2),
    torch.nn.Linear(15, 20),
    torch.nn.LeakyReLU(0.2),
    torch.nn.Linear(20, INPUT_DIM),
]

DECODER_MODEL = TorchPerceptronModel(2, INPUT_DIM, DECODE_LAYERS)

VAE_MODEL = TorchVAEModel(ENCODER_MODEL, DECODER_MODEL)

VAE_ALGORITHM = TorchVAEAlgorithm(
    interface=VAE_MODEL
)


In [6]:
# Now we make the algorithm
VAE_ALGORITHM.train(DATASET, epochs=100)

[2025-08-08 11:28:06] INFO     ┌──────────────────────────────────┐                                    ]8;id=226722;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=283003;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#111\111]8;;\

                      INFO     │ PyTorch Training Configuration │                                      ]8;id=636987;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=576797;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#112\112]8;;\

                      INFO     └──────────────────────────────────┘                                    ]8;id=607813;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=612377;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#113\113]8;;\

[2025-08-08 11:28:12] INFO     ┌─────────────────────┐                                                 ]8;id=84131;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=767676;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#111\111]8;;\

                      INFO     │ Training Progress │                                                   ]8;id=47967;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=245320;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#112\112]8;;\

                      INFO     └─────────────────────┘                                                 ]8;id=929992;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=335231;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#113\113]8;;\

  0%|          | 0/100 [00:00<?, ?it/s]

[2025-08-08 11:28:40] INFO     ✅ Training completed! Final loss: 6281.443359 | Total epochs: 100      ]8;id=609183;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=754547;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#125\125]8;;\

In [8]:
import numpy as np
import uproot

new_file = uproot.recreate("tmp.root")
new_file["posteriors"] = {
    "xsec_1": np.linspace(0,2,
    "xsec_2": np.random.rand(1000).astype(np.float32),
    "xsec_3": np.random.rand(1000).astype(np.float32),
}

new_file.close()
test_frame = ROOTDataset("tmp.root", tree_name="posteriors", branches=["xsec_1", "xsec_2", "xsec_3"], labels=["xsec_1", "xsec_2", "xsec_3"])

data, labels = test_frame.get_items(0, len(test_frame))
preds = VAE_ALGORITHM.predict(test_frame)

print(preds)

[2025-08-08 11:28:49] INFO     ┌────────────────────────────┐                                          ]8;id=811276;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=872139;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#111\111]8;;\

                      INFO     │ Initializing ROOTDataset │                                            ]8;id=791378;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=386661;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#112\112]8;;\

                      INFO     └────────────────────────────┘                                          ]8;id=861024;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=178777;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#113\113]8;;\

                      INFO     ℹ️ Opening ROOT file: tmp.root                                           ]8;id=638950;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=843001;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#169\169]8;;\

                      INFO     ℹ️ Tree 'posteriors' has 1000 entries                                    ]8;id=519812;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=474971;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#169\169]8;;\

...

                      INFO     ℹ️ Found 1000 unique entries after applying filters (100.00%% of total)  ]8;id=558757;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=801336;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#169\169]8;;\

                      INFO     ✅ Initialized ROOTDataset with 1000 entries, branches: ['xsec_1',      ]8;id=711935;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=609960;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#125\125]8;;\
                               'xsec_2', 'xsec_3'], labels: ['xsec_1', 'xsec_2', 'xsec_3']                         

                      INFO     ℹ️ 🔮 Running predictions on dataset...                                  ]8;id=990351;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=936161;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#169\169]8;;\

                      INFO     ✅ Predictions completed!                                               ]8;id=110071;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py\utils.py]8;;\:]8;id=533536;file:///home/henryi/sft/MaCh3-PythonUtils/src/mach3pythonutils/utils/utils.py#125\125]8;;\

(tensor([[0.9165, 0.9109, 1.0110],
        [0.9186, 0.9276, 0.9814],
        [0.9339, 0.9273, 1.0364],
        ...,
        [0.9977, 1.0017, 1.0545],
        [1.1517, 1.2346, 1.2117],
        [1.1049, 1.1481, 1.1465]]), tensor([[-0.0922,  0.1226],
        [-0.1134,  0.1394],
        [-0.0936,  0.1274],
        ...,
        [-0.0833,  0.1115],
        [-0.0708,  0.1001],
        [-0.1278,  0.1516]]), tensor([[-7.7817e-05, -1.0636e-01],
        [ 2.0282e-03, -1.3911e-01],
        [ 4.5351e-03, -8.7669e-02],
        ...,
        [-2.2523e-03, -1.0740e-01],
        [-3.8119e-03, -9.3063e-02],
        [ 4.6691e-03, -1.5292e-01]]))
